# AI Resume Screener — Model Training Pipeline
### TF-IDF + LightGBM with K-Fold Cross Validation
This notebook trains a classifier to predict resume-job match quality (A/B/C/D grade).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')
from app.utils.preprocessor import preprocess

print('All imports OK')

## 1. Load Dataset

In [ ]:
# ---------- Synthetic training data (replace with real data) ----------
# Label: A=Strong Match, B=Good, C=Partial, D=Weak

samples = [
    # (resume_text, jd_text, label)
    ('python machine learning scikit-learn pandas numpy tensorflow deep learning nlp flask rest api', 
     'python machine learning nlp data science tensorflow keras', 'A'),
    ('react javascript node express mongodb html css bootstrap frontend web developer',
     'react javascript frontend web developer html css', 'A'),
    ('java springboot microservices mysql rest api agile docker kubernetes',
     'java springboot backend microservices rest api', 'A'),
    ('python data analysis pandas sql database reporting',
     'machine learning deep learning tensorflow pytorch nlp', 'C'),
    ('html css javascript basic web design photoshop',
     'react node.js full stack developer microservices docker', 'D'),
    ('aws cloud architect terraform kubernetes docker ci/cd devops jenkins',
     'aws cloud devops kubernetes docker terraform', 'A'),
    ('android java kotlin mobile firebase sqlite',
     'python flask machine learning api data science', 'D'),
    ('python flask django rest api postgresql docker linux',
     'flask python rest api backend web developer', 'A'),
    ('data science python r statistics machine learning regression classification',
     'data analyst sql excel power bi tableau reporting', 'B'),
    ('sql mysql reporting excel power bi business analyst',
     'sql data analyst reporting power bi dashboard', 'A'),
    ('c++ embedded systems firmware iot arduino raspberry pi',
     'python web developer flask django rest api', 'D'),
    ('java python algorithms data structures competitive programming leetcode',
     'software engineer algorithms java problem solving', 'B'),
    ('deep learning computer vision opencv pytorch yolo image classification',
     'machine learning python computer vision opencv', 'A'),
    ('hadoop spark kafka hive big data pipeline etl data engineering',
     'data engineer apache spark kafka hadoop pipeline', 'A'),
    ('javascript react vue angular typescript frontend ui ux design',
     'full stack developer node react postgresql devops', 'B'),
]

df = pd.DataFrame(samples, columns=['resume', 'jd', 'label'])

# Combine resume + JD as input feature
df['combined'] = df['resume'].apply(preprocess) + ' SEP ' + df['jd'].apply(preprocess)

print(df['label'].value_counts())
df.head()

## 2. TF-IDF Feature Extraction

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=3000, sublinear_tf=True)
X = tfidf.fit_transform(df['combined'])

le = LabelEncoder()
y = le.fit_transform(df['label'])

print(f'Feature matrix shape: {X.shape}')
print(f'Classes: {le.classes_}')

## 3. LightGBM with Stratified K-Fold Cross Validation

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cv_scores = cross_val_score(lgb_model, X, y, cv=skf, scoring='accuracy')

print(f'CV Accuracy per fold: {cv_scores}')
print(f'Mean CV Accuracy:     {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 4. Train Final Model & Evaluate

In [ ]:
lgb_model.fit(X, y)

y_pred = lgb_model.predict(X)

print('Classification Report:')
print(classification_report(y, y_pred, target_names=le.classes_))

# Confusion Matrix
cm = confusion_matrix(y, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=120)
plt.show()

## 5. Feature Importance

In [ ]:
feature_names = tfidf.get_feature_names_out()
importances   = lgb_model.feature_importances_

top_n = 20
top_idx = np.argsort(importances)[-top_n:][::-1]

plt.figure(figsize=(8, 5))
plt.barh(range(top_n), importances[top_idx], color='#2563eb')
plt.yticks(range(top_n), feature_names[top_idx])
plt.xlabel('Importance')
plt.title(f'Top {top_n} TF-IDF Features (LightGBM)')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=120)
plt.show()

## 6. Save Model & Vectorizer

In [ ]:
import os
os.makedirs('../app/models', exist_ok=True)

joblib.dump({'model': lgb_model, 'vectorizer': tfidf, 'label_encoder': le},
            '../app/models/resume_model.pkl')

print('Model saved to app/models/resume_model.pkl')

## 7. Test Inference

In [ ]:
saved    = joblib.load('../app/models/resume_model.pkl')
model_   = saved['model']
tfidf_   = saved['vectorizer']
le_      = saved['label_encoder']

test_resume = 'python flask machine learning nlp sklearn pandas rest api'
test_jd     = 'python developer flask api machine learning data science'

combined = preprocess(test_resume) + ' SEP ' + preprocess(test_jd)
X_test   = tfidf_.transform([combined])
pred     = model_.predict(X_test)
proba    = model_.predict_proba(X_test)

print(f'Predicted Grade : {le_.inverse_transform(pred)[0]}')
print(f'Class Probabilities: {dict(zip(le_.classes_, proba[0].round(3)))}')